# 7) Mini-batch training of MLP


> Phases 1) 2) 3) are VERY STANDARD and VERY FUNDAMENTAL / REPEATED in this course

> In 4) there is the alternative way of training a MLP model with one_hot_encoding of labels, which is actually suggested only for categorical labels (not our case).

----

>**Mini-batch training**: *mini-batch training comes naturally when using DataLoader for batching on the training set.*
>
> - The training dataset is divided into small random subsets of the same size called mini-batches, by *DataLoader*. 
> - The training phase is performed on each of the mini-batches, one at a time: output prediction, loss computation, gradient computation, parameters updating. Until all mini-batch are visited.

- This method increases the randomness, and avoids that the loss gets stucked into local minima or flat regions.
- We can choose the loss we want.
- We can choose the optimization method we want on the mini-batches: SGD or Adam.
- *The combination of mini-batch training + Adam gives the best results.*

-----

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import MNIST

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## 1) Prepare the training and validation datasets with batching

1) class Dataset
2) split dataset with Subsets
3) class DataLoader for batching

In [3]:
# 1) class Dataset

class MNISTDataset(Dataset):                                    # dataset constructor

    def __init__(self):

        data = MNIST('mnist', download=True)
        self.images, self.labels = data.data, data.targets      # return dataset X, y


    def __len__(self):

        return len(self.images)                                 # number of features

    
    def __getitem__(self, idx: int):                            # return element indexed idx in X, y

        image = self.images[idx].float().view(28*28) / 255.0
        label = self.labels[idx]

        return image, label
        

In [4]:
# call/create dataset

dataset = MNISTDataset()    

In [5]:
# 2) create training and validation datasets with Subsets

idx_train, idx_valid = train_test_split(range(len(dataset)), test_size=0.1, random_state=0)

train_dataset = Subset(dataset, idx_train)        
valid_dataset = Subset(dataset, idx_valid)

In [6]:
# OPTIONAL ARGUMENT in DataLoader object,
#
# TELLING HOW TO FORM THE BATCH
# 
# The default is to stock data as Torch tensors, which is exactly what this function does

def collate_fn(batch):                      # returns X, y in batch

    images = []
    labels = []

    for image, label in batch:
        images.append(image)
        labels.append(label)

    images = torch.stack(images, dim=0)     # size : (32, 784)
    labels = torch.tensor(labels)           # size : (32,)

    return images, labels


In [7]:
# 3) create training and validation datasets with batching, with DataLoader

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True,
    collate_fn=collate_fn
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    drop_last=False
)

## 2) Define the MLP classification model

Complete and standard construction of a MLP classifier.

In [8]:
#class constructor of MLP multiclassifier

class TorchMLP(torch.nn.Module):

    def __init__(self, input_dim, hidden_dims, num_classes):
        super().__init__()
        
        all_dims = [input_dim] + list(hidden_dims) + [num_classes]
        
        layers = []
        for dim1, dim2 in zip(all_dims[:-1], all_dims[1:]):
            layers.append(torch.nn.Linear(dim1, dim2))
            layers.append(torch.nn.ReLU())
            
        layers = layers[:-1]                            # take off ReLu at last

        self.model = torch.nn.Sequential(*layers)       # model where layers are called sequentially
        
    def forward(self, x):
        return self.model(x)


In [ ]:
# CALL MODEL, LOSS, OPTIMIZER ADAM

modele = TorchMLP(784, [100, 200, 100], 10)#.cuda()

loss_function = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(modele.parameters(), lr=0.002) # or other optimizers


## 3) Train and evaluate the MLP model classifier with batching

4) Use the training and validation datasets obtained from batching

In [ ]:
# TRAINING AND VALIDATION OF MLP

for epoch in range(10):

    # TRAINING
    for images, labels in train_dataloader:    # mini-batch training, for natural

        optimizer.zero_grad()
        y_pred = modele(images)
        loss_result = loss_function(y_pred, labels)
        loss_result.backward()
        optimizer.step()

    # EVALUATION 
    all_preds = []
    all_labels = []

    with torch.no_grad():                                  # cancel gradient, to save memory

        for images, labels in valid_dataloader:            # in validation set, compute accuracy

            logits = modele(images)
            all_preds.extend(torch.argmax(logits, dim=1))
            all_labels.extend(labels)

    print(accuracy_score(all_labels, all_preds))
 

0.9478333333333333
0.964
0.9711666666666666
0.9711666666666666
0.9708333333333333
0.972
0.9766666666666667
0.9778333333333333
0.9753333333333334
0.9768333333333333


## 4) Alternative version of training: training with **one_hot_encoding** of labels

>**One Hot Encoding**: *encode a class label into a vector containing all zeros and a single one*. More generally, it encodes all labels into a matrix of zeros, where each rows has a single 1 encoding the corresponding label.

For example, if the MLP model classifies data into 10 classes,

$$
labels = [3, 7, 1]
$$

can be one_hot_encoded into

$$
[[0, 0, 0, 1, 0, 0, 0, 0, 0, 0], \\
 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0], \\
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0]]. \\
$$

- The label 3 is one_hot_encoded into [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
- The label 7 is one_hot_encoded into [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
- The label 1 is one_hot_encoded into [0, 1, 0, 0, 0, 0, 0, 0, 0, 0]

------

The code 


''one_hot_labels = torch.zeros((len(labels), 10), dtype=torch.float32)''


creates a matrix of all zero with 

- rows == number of labels, i.e. |y_labels|
- columns == number of classes, i.e. 10.

Then, the code

''one_hot_labels[torch.arange(len(labels)), labels] = 1 ''

for each row, sets a single 1 to encode a label.

As this, *we obtain a matrix where each row represents a single hot_one_encoded label*.

-------

>**When one_hot_encoding of labels is useful?** *When labels are categorical*; to allow the loss to take in input categorical labels.

In our specific case, labels are already numbers, therefore the one_hot_encoding is useless: the cross-entropy loss is perfectly able to process the data labels as numbers. *If the labels are already numerical, it is preferable to not one_hot_encoding and take it simple.*

Rmk) one_hot_encoding concerns only the training phase!! and not the evaluation phase.

In [ ]:
for epoch in range(10):

    # TRAINING
    for images, labels in train_dataloader:

        optimizer.zero_grad()
        y_pred = modele(images)#.to('cuda'))

        # ONE_HOT_ENCODING OF LABELS
        one_hot_labels = torch.zeros((len(labels), 10), dtype=torch.float32)   # produces a matrix of zeroes for 10 classes
        one_hot_labels[torch.arange(len(labels)), labels] = 1                  # encode the labels into vectors

        loss_result = loss_function(y_pred, one_hot_labels)#.to('cuda'))
        
        loss_result.backward()
        optimizer.step()

    # EVALUATION
    all_preds = []
    all_labels = []
    with torch.no_grad():

        for images, labels in valid_dataloader:

            logits = modele(images)#.to('cuda'))
            all_preds.extend(torch.argmax(logits, dim=1).to('cpu'))
            all_labels.extend(labels)
            
    print(accuracy_score(all_labels, all_preds))

0.977
0.9763333333333334
0.9765
0.9746666666666667
0.9758333333333333
0.9801666666666666
0.9815
0.9748333333333333
0.9753333333333334
0.9746666666666667
